# 07d — Historical Context Enrichment

Primary: exact-year CLCD physical context. Optional: historical OSM/ohsome cross-check.

This stage does not infer occupation, WORK, or OFFICE.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path('/root/geolife')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]','rasterio>=1.4,<2'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())


In [ ]:
from importlib.util import spec_from_file_location,module_from_spec
import pandas as pd
from geolife.model import HomeOfficeConfig, build_semantic_locations
def load(name,path):
    spec=spec_from_file_location(name,path); m=module_from_spec(spec); sys.modules[name]=m; spec.loader.exec_module(m); return m
s07d=load('s07d',REPO_DIR/'analysis'/'07d_historical_context_enrichment.py')
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07d_historical_context_enrichment'; C.mkdir(parents=True,exist_ok=True)
RAW_CACHE=V/'cache'/'07d_historical_context_enrichment'/'ohsome_raw'; RAW_CACHE.mkdir(parents=True,exist_ok=True)
ANCHORS=V/'cache'/'cp2_v2'/'07c_historical_source_alignment'/'anchor_observation_dates_private.pkl'
if not ANCHORS.exists(): raise FileNotFoundError(f'Run CP2-v2 Stage 07c first: {ANCHORS}')
anchors=pd.read_pickle(ANCHORS)
expected_namespace='production_complete_link_200m_all_resolved_timezone_v2'
if 'location_namespace' not in anchors.columns or not anchors['location_namespace'].eq(expected_namespace).all():
    raise ValueError('Stage 07c artifact uses an incompatible location namespace; rerun CP2-v2 07c first')

if not {'latitude','longitude'}.issubset(anchors.columns):
    stay_paths=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if not stay_paths: raise FileNotFoundError('stays_baseline_v1.pkl is required to reconstruct production anchor coordinates')
    stays=pd.read_pickle(stay_paths[0])
    cfg=HomeOfficeConfig(clustering_method='complete_link',location_max_diameter_m=200.0)
    _,locations=build_semantic_locations(stays,config=cfg)
    anchors=s07d.attach_anchor_coordinates_from_locations(anchors,locations)
    print('attached private coordinates from CP2-v2 production semantic locations')

print('anchors:',len(anchors),'users:',anchors['user_id'].astype(str).nunique())
print('location namespace:',anchors['location_namespace'].iloc[0])
print('coordinate columns present:', {'latitude','longitude'}.issubset(anchors.columns))
print('shared historical OSM raw cache:', RAW_CACHE)
display(s07d.build_clcd_sampling_plan(anchors))


## CLCD exact-year physical context

The runner first checks `/mnt/geolife-data/external/clcd/CLCD_v01_<year>.tif`. If absent, it remote-opens the pinned Zenodo Cloud Optimized GeoTIFF.


In [ ]:
LOCAL_CLCD=V/'external'/'clcd'
url_by_year={}
for year in sorted(pd.to_numeric(anchors['median_observation_year']).astype(int).unique()):
    local=LOCAL_CLCD/f'CLCD_v01_{year}_albert.tif'
    if local.exists(): url_by_year[int(year)]=str(local)
print('local CLCD files used:',len(url_by_year))
clcd=s07d.sample_clcd_remote(anchors,url_by_year=url_by_year or None,local_windows=(3,5))
assert len(clcd)==len(anchors), (len(clcd),len(anchors))
print('CLCD POPULATION SUMMARY'); display(s07d.summarize_clcd_population(clcd))
print('CLCD CLASS SUMMARY'); display(s07d.summarize_clcd(clcd))


## Historical OSM cross-check via Modal Secret — free-tier safe

The notebook uses the Modal secret `ohsome-api` containing `OHSOME_API_KEY`.

Free-tier behavior is intentionally conservative:

- cached anchor responses are reused;
- only a bounded number of fresh requests are attempted per run;
- requests are sequential with a configurable pause;
- the first HTTP 429 stops new requests for that run;
- partial ohsome results are saved and the notebook continues;
- rerunning later resumes from cache instead of restarting.


In [ ]:
import modal

ohsome_eligible = s07d.ohsome_eligible_anchors(anchors)
ohsome_ineligible = anchors.loc[~anchors.index.isin(ohsome_eligible.index)].copy()
print('ohsome eligible anchors:', len(ohsome_eligible), '/', len(anchors))
print('ohsome ineligible anchors:', len(ohsome_ineligible))

ohsome_app = modal.App('geolife-stage07d-ohsome')
ohsome_secret = modal.Secret.from_name(
    'ohsome-api',
    required_keys=['OHSOME_API_KEY'],
)
ohsome_image = modal.Image.debian_slim(
    python_version=f'{sys.version_info.major}.{sys.version_info.minor}'
).pip_install('httpx')

@ohsome_app.function(
    image=ohsome_image,
    secrets=[ohsome_secret],
    serialized=True,
    timeout=300,
)
def _ohsome_post_with_secret(body: dict, api_url: str) -> dict:
    import os
    import httpx
    try:
        response = httpx.post(
            api_url.rstrip('/') + '/extraction/features.parquet',
            json=body,
            headers={'authorization': os.environ['OHSOME_API_KEY']},
            timeout=240,
        )
    except Exception as exc:
        return {
            'ok': False,
            'status_code': 0,
            'content': None,
            'retry_after': None,
            'rate_limit_reset': None,
            'error': f'{type(exc).__name__}: {exc}',
        }
    headers = response.headers
    if response.status_code >= 400:
        return {
            'ok': False,
            'status_code': int(response.status_code),
            'content': None,
            'retry_after': headers.get('Retry-After'),
            'rate_limit_reset': headers.get('RateLimit-Reset') or headers.get('X-RateLimit-Reset'),
            'error': response.text[:500],
        }
    return {
        'ok': True,
        'status_code': int(response.status_code),
        'content': response.content,
        'retry_after': headers.get('Retry-After'),
        'rate_limit_reset': headers.get('RateLimit-Reset') or headers.get('X-RateLimit-Reset'),
        'error': None,
    }

api_url = os.environ.get('OHSOME_API_URL', s07d.OHSOME_API_DEFAULT)
max_new_requests = int(os.environ.get('OHSOME_MAX_NEW_REQUESTS_PER_RUN', '20'))
pause_s = float(os.environ.get('OHSOME_REQUEST_PAUSE_S', '6'))

ohsome = pd.DataFrame()
request_log = pd.DataFrame()
with ohsome_app.run():
    def modal_fetcher(body: dict):
        return _ohsome_post_with_secret.remote(body, api_url)

    ohsome, request_log = s07d.fetch_ohsome_context_with_fetcher(
        ohsome_eligible,
        fetcher=modal_fetcher,
        cache_dir=RAW_CACHE,
        radius_m=100,
        pause_s=pause_s,
        max_new_requests_per_run=max_new_requests,
    )

ohsome_summary = s07d.summarize_ohsome(ohsome)
ohsome_run_summary = s07d.summarize_ohsome_run(ohsome_eligible, ohsome, request_log)
ohsome_run_summary['all_anchor_count'] = len(anchors)
ohsome_run_summary['ineligible_pre_ohsome'] = len(ohsome_ineligible)
print('OHSOME RUN SUMMARY')
display(ohsome_run_summary)
print('OHSOME CONTEXT SUMMARY — completed eligible anchors only')
display(ohsome_summary)
if not request_log.empty and request_log['status'].eq('rate_limited').any():
    cols=['status','retry_after','rate_limit_reset','error']
    print('Rate limit hit; rerun later to resume from cache.')
    display(request_log.loc[request_log['status'].eq('rate_limited'), cols].head(1))


In [ ]:
historical=s07d.join_historical_context(anchors,clcd,ohsome if not ohsome.empty else None)
clcd.to_pickle(C/'clcd_anchor_context_private.pkl')
historical.to_pickle(C/'historical_context_private.pkl')
if not ohsome.empty: ohsome.to_pickle(C/'ohsome_anchor_context_private.pkl')
if not request_log.empty: request_log.to_pickle(C/'ohsome_request_log_private.pkl')
s07d.build_clcd_sampling_plan(anchors).to_csv(C/'clcd_sampling_plan.csv',index=False)
s07d.summarize_clcd_population(clcd).to_csv(C/'clcd_population_summary.csv',index=False)
s07d.summarize_clcd(clcd).to_csv(C/'clcd_class_summary.csv',index=False)
ohsome_summary.to_csv(C/'ohsome_summary.csv',index=False)
ohsome_run_summary.to_csv(C/'ohsome_run_summary.csv',index=False)
if not request_log.empty: request_log.to_csv(C/'ohsome_request_log.csv',index=False)
print('saved:',C)


## Boundary

- CLCD is physical land cover only; `impervious` is not office/residential/WORK.
- point vs 3×3 / 5×5 agreement is spatial sensitivity, not confidence calibration.
- ohsome is a historical mapping cross-check; missing OSM is not proof of real-world absence.
- BCL POI 2008 remains blocked until exact file access/license/CRS are verified.
